# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedRamadan164/FlyRank_ML_internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MohamedRamadan164/FlyRank_ML_internship"
REPO_DIR = "FlyRank_ML_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("rows:", len(df), "| clients:", df["client_id"].nunique())


rows: 30000 | clients: 32


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

**Answer.** **Refresh / Content Opportunity Scoring.** Every earlier notebook in this repo (`w02`–`w05`) worked the signal side of this lane's sibling — which safe signals correlate with a page's trend. That groundwork turns directly into this lane's real deliverable: instead of just describing signals, score every content item and output a **ranked, reason-coded action queue** (`protect` / `refresh` / `monitor`) that a content team could actually work from. It's the lane where the earlier weeks' evidence — staleness is a MIXED signal alone, CTR vs. position is CONFIRMED, combined signals clear a 0.70+ AUC — turns into a decision, not just an observation.

In [2]:
print("Lane: Refresh / Content Opportunity Scoring")
print("Builds on: w02 (task framing) -> w03 (data contract) -> w04 (baseline rule) -> w05 (model vs baseline)")


Lane: Refresh / Content Opportunity Scoring
Builds on: w02 (task framing) -> w03 (data contract) -> w04 (baseline rule) -> w05 (model vs baseline)


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Decision:** which of a client's content items should a content editor spend their limited review time on this month? **Who acts:** a content/SEO editor working a fixed-size review queue, not an automated system. **Action:** the editor opens the top-ranked pages and decides whether to rewrite, update, or leave them — the model never publishes anything itself. **Cost of a wrong call:** two different costs, and they're not symmetric. A **false positive** (flagged for refresh, actually fine) costs an editor's time on a page that didn't need it — annoying, but cheap and reversible. A **false negative** (a genuinely declining page never surfaced) costs a real page silently losing traffic with nobody looking — worse, because nothing prompts anyone to check. That asymmetry is why the queue is ranked (editors work top-down until their time runs out) rather than a hard yes/no cutoff.

In [3]:
print("Decision: which content items get a human review this month")
print("Actor: content/SEO editor with a fixed-size queue")
print("FP cost: wasted review time (cheap, reversible)")
print("FN cost: a real decline goes unnoticed (worse, silent)")


Decision: which content items get a human review this month
Actor: content/SEO editor with a fixed-size queue
FP cost: wasted review time (cheap, reversible)
FN cost: a real decline goes unnoticed (worse, silent)


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

**Answer.** On the 30,000-row starter sample: 54.2% of content items are currently in the "down" trend bucket — more than half, so a review queue has real work to do. A simple stale-and-still-visible gate (updated 90+ days ago, impressions above the sample median) already narrows that to 5,992 items (19.9% of the sample) — a workable-sized queue instead of 30,000 undifferentiated rows. And per `w04`'s real signal check, CTR clearly separates by position tier (top_3 ≈0.49% down to deep ≈0.04%, weighted, n in the thousands per tier) — there's real, checkable structure here, not noise to force a model onto.

In [4]:
decline_rate = (df["trend_direction"] == "down").mean()
gate = (df["days_since_last_update"] >= 90) & (df["impressions_90d"] >= df["impressions_90d"].median())

print(f"Share currently 'down': {decline_rate:.1%}")
print(f"Stale+visible gate narrows 30,000 rows to: {gate.sum()} ({gate.mean():.1%})")

valid = df[df["avg_position"] > 0]
ctr_by_tier = valid.groupby("position_tier").apply(
    lambda s: 100 * s["clicks_90d"].sum() / s["impressions_90d"].sum()
).round(3)
print("\nWeighted CTR by position tier:")
print(ctr_by_tier)


Share currently 'down': 54.2%
Stale+visible gate narrows 30,000 rows to: 5992 (20.0%)

Weighted CTR by position tier:
position_tier
deep        0.041
page_1      0.350
page_3_5    0.155
striking    0.347
top_3       0.489
dtype: float64


/tmp/ipykernel_3180/1013927076.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ctr_by_tier = valid.groupby("position_tier").apply(


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**Can say:** this queue is **decision-support** — it ranks content by an observed, directional association between signals (staleness, CTR-vs-position, volume) and a same-window trend proxy, validated against a rule baseline on a held-out, client-grouped split. Confidence in any single row is bounded by the base rate and the model's own precision@K on that split — nothing here is a guarantee about one specific page.

**Can never say:** that any recommendation *causes* a ranking or traffic change once acted on (no experiment, no before/after causal design exists in this data); that the model "predicts Google's algorithm" (it predicts a proxy label computed from the same panel, not the search engine); or that a page flagged `protect`/`refresh` will behave that way with certainty — every output is a probability-ranked suggestion for a human to review, not an automated verdict.

In [5]:
print("Claim type: decision-support, observed + directional association, validated vs a rule baseline")
print("Never: causal impact of acting on a recommendation, or a claim about Google's algorithm itself")


Claim type: decision-support, observed + directional association, validated vs a rule baseline
Never: causal impact of acting on a recommendation, or a claim about Google's algorithm itself


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.